# Table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

Deals  = pd.read_parquet('/content/drive/MyDrive/fin_project/Deals_clean.parquet')
Spend  = pd.read_parquet('/content/drive/MyDrive/fin_project/Spend_clean.parquet')
Contacts = pd.read_parquet('/content/drive/MyDrive/fin_project/Contacts_clean.parquet')

# Overall Unit Economics

In [ ]:
PATH = '/content/drive/MyDrive/fin_project/'

Deals    = pd.read_parquet(PATH + 'Deals_clean.parquet')
Spend    = pd.read_parquet(PATH + 'Spend_clean.parquet')
Contacts = pd.read_parquet(PATH + 'Contacts_clean.parquet')

# ============================================================
# Payment Done deals only
# ============================================================

paid_mask = (
    (Deals['Stage'] == 'Payment Done') &
    (Deals['Initial Amount Paid clean'] > 1) &
    (~Deals['Product'].isin(['Find yourself in IT', 'Data Analytics']))
)

paid_deals = Deals[paid_mask].copy()

# ============================================================
# Unit Economics calculation function
# ============================================================

def print_unit_economics(label, deals_subset, contacts_count, spend_df):

    UA = contacts_count

    # Buyers (deduplicated by deal Id)
    buyers = deals_subset.drop_duplicates(subset=['Id'])
    B = len(buyers)

    # Transactions:
    # Installments → number of paid months (Months of study)
    # One-time → 1 transaction per deal
    transactions = int(
        deals_subset.loc[
            deals_subset['Payment Type'] == 'Recurring Payments',
            'Months of study'
        ].sum()
        +
        (deals_subset['Payment Type'] == 'One Payment').sum()
    )

    # Conversion
    C1 = B / UA * 100 if UA > 0 else 0

    # Marketing
    AC           = spend_df['Spend'].sum()
    total_clicks = spend_df['Clicks'].sum()
    CPA          = AC / UA if UA > 0 else 0
    CPC          = AC / total_clicks if total_clicks > 0 else 0
    CAC          = AC / B if B > 0 else 0

    # Financials
    revenue      = deals_subset['Revenue'].sum()
    contract_rev = deals_subset['Offer Total Amount clean'].sum()
    unrealized   = contract_rev - revenue

    # AOV = average revenue per transaction
    AOV = revenue / transactions if transactions > 0 else 0

    # APC = average number of payments per buyer
    APC = transactions / B if B > 0 else 0

    # Average number of study months
    avg_months = deals_subset['Months of study'].mean()

    # COGS not present in the data
    COGS = 0

    # CLTV per the instructor's methodology
    CLTV = (AOV - COGS) * APC

    # LTV
    LTV = CLTV * (C1 / 100)

    # Contribution Margin
    CM = revenue - AC

    # Contribution Margin per buyer
    CM1 = CLTV - CAC

    print("=" * 70)
    print(f"{label}")
    print("=" * 70)
    print(f"{'UA (total leads)':<40}: {UA:>10,.0f}")
    print(f"{'B (buyers)':<40}: {B:>10,.0f}")
    print(f"{'Transactions':<40}: {transactions:>10,.0f}")
    print(f"{'C1 (%)':<40}: {C1:>10.2f}%")
    print(f"{'AC (€)':<40}: € {AC:>9,.0f}")
    print()
    print(f"{'Revenue (€)':<40}: € {revenue:>9,.0f}")
    print(f"{'Contract Revenue (€)':<40}: € {contract_rev:>9,.0f}")
    print(f"{'Unrealized Revenue (€)':<40}: € {unrealized:>9,.0f}")
    print()
    print(f"{'APC (Transactions / Buyer)':<40}: {APC:>10.2f}")
    print(f"{'Avg Months':<40}: {avg_months:>10.2f}")
    print(f"{'AOV (Revenue / Transaction, €)':<40}: € {AOV:>9,.0f}")
    print(f"{'CAC (€)':<40}: € {CAC:>9,.0f}")
    print(f"{'CLTV (€)':<40}: € {CLTV:>9,.0f}")
    print(f"{'LTV (€)':<40}: € {LTV:>9,.0f}")
    print(f"{'CM (€)':<40}: € {CM:>9,.0f}")
    print(f"{'CM1 (€)':<40}: € {CM1:>9,.0f}")
    print(f"{'CPA (€)':<40}: € {CPA:>9.2f}")
    print(f"{'CPC (€)':<40}: € {CPC:>9.2f}")
    print()

# ============================================================
# Overall Unit Economics
# ============================================================

UA_total = len(Contacts)

print_unit_economics(
    "GLOBAL UNIT ECONOMICS",
    paid_deals,
    UA_total,
    Spend
)

# ============================================================
# By product
# ============================================================

products = [
    'Digital Marketing',
    'UX/UI Design',
    'Web Developer'
]

for product in products:

    subset = paid_deals[
        paid_deals['Product'] == product
    ]

    print_unit_economics(
        f"PRODUCT: {product}",
        subset,
        UA_total,
        Spend
    )
    # 839 — unique deals (one person can buy twice). By business logic, 839 is correct — each payment is a separate transaction.

In [ ]:
mask = (
    (Deals['Stage'] == 'Payment Done') &
    (Deals['Initial Amount Paid clean'] > 1) &
    (~Deals['Product'].isin(['Find yourself in IT', 'Data Analytics']))
)

paid = Deals[mask].copy()

UA     = len(Contacts)
AC     = Spend['Spend'].sum()
Clicks = Spend['Clicks'].sum()

def calc_metrics(
    label,
    deals_subset,
    ua,
    ac,
    clicks,
    c1_mult=1.0,
    apc_mult=1.0,
    revenue_mult=1.0,
    cac_mult=1.0,
    avg_months_mult=1.0  # new parameter for H3
):
    # Buyers
    buyers       = deals_subset.drop_duplicates(subset=['Id'])
    students     = len(buyers)
    students_adj = round(students * c1_mult)

    # Avg Months — grows as retention grows
    avg_months = deals_subset['Months of study'].mean() * avg_months_mult

    # Transactions:
    # As retention grows → Months of study grow → more recurring transactions
    transactions_recurring = (
        deals_subset.loc[
            deals_subset['Payment Type'] == 'Recurring Payments',
            'Months of study'
        ].sum() * avg_months_mult
    )
    transactions_one = (deals_subset['Payment Type'] == 'One Payment').sum()
    transactions = round(
        (transactions_recurring + transactions_one) * c1_mult * revenue_mult
    )

    # Conversion
    c1 = students_adj / ua * 100 if ua > 0 else 0

    # Revenue — grows from retention via more months of study
    revenue    = deals_subset['Revenue'].sum() * c1_mult * apc_mult * revenue_mult * avg_months_mult
    contract   = deals_subset['Offer Total Amount clean'].sum()
    unrealized = contract - revenue

    # APC = Transactions / Buyers (number of transactions per buyer)
    apc = transactions / students_adj if students_adj > 0 else 0

    # AOV = Revenue / Transactions
    aov = revenue / transactions if transactions > 0 else 0

    # CLTV = AOV × APC
    cltv = aov * apc

    # LTV = CLTV × C1
    ltv = cltv * (c1 / 100)

    # CAC, CPA, CPC
    cac = (ac / students_adj) * cac_mult if students_adj > 0 else 0
    cpa = ac / ua if ua > 0 else 0
    cpc = ac / clicks if clicks > 0 else 0

    # CM, CM1
    cm  = revenue - ac
    cm1 = cltv - cac

    return {
        'Scenario':             label,
        'UA':                   round(ua),
        'Students':             students_adj,
        'Transactions':         transactions,
        'C1_%':                 round(c1, 2),
        'Revenue (€)':          round(revenue),
        'Contract Revenue (€)': round(contract),
        'Unrealized (€)':       round(unrealized),
        'APC':                  round(apc, 2),
        'Avg Months':           round(avg_months, 2),
        'AOV (€)':              round(aov),
        'CLTV (€)':             round(cltv),
        'LTV (€)':              round(ltv),
        'CAC (€)':              round(cac),
        'CPA (€)':              round(cpa, 2),
        'CPC (€)':              round(cpc, 2),
        'CM (€)':               round(cm),
        'CM1 (€)':              round(cm1),
    }

# ============================================================
# HYPOTHESES
#
# H0: Conversion stays at the current level.
# H1: Conversion increases by 10%.
#
# H0: CAC does not change.
# H1: CAC decreases by 10%.
# ============================================================

scenarios = [

    ('Baseline', {}),

    ('H1: Conversion +10%', {
        'c1_mult': 1.10
    }),

    ('H2: CAC -10%', {
        'c1_mult': 1.10
    })

]

# GLOBAL
global_rows = []
for label, params in scenarios:
    global_rows.append(calc_metrics(label, paid, UA, AC, Clicks, **params))

df_global = pd.DataFrame(global_rows)
df_global.insert(0, 'Product', 'All products')

print("=" * 100)
print("GLOBAL SCENARIOS")
print("=" * 100)
display(df_global)

# PRODUCTS
product_rows = []
for product in ['Digital Marketing', 'UX/UI Design', 'Web Developer']:
    subset = paid[paid['Product'] == product]
    for label, params in scenarios:
        row = calc_metrics(label, subset, UA, AC, Clicks, **params)
        row['Product'] = product
        product_rows.append(row)

df_products = pd.DataFrame(product_rows)
cols = ['Product'] + [c for c in df_products.columns if c != 'Product']
df_products = df_products[cols]

print("=" * 100)
print("PRODUCT SCENARIOS")
print("=" * 100)
display(df_products)

# ALL TABLE
df_all = pd.concat([df_global, df_products], ignore_index=True)
df_all['Hypothesis'] = [
    'Baseline' if 'Baseline' in s else
    'H1' if 'H1' in s else
    'H2'
    for s in df_all['Scenario']
]

print("=" * 100)
print("ALL SCENARIOS")
print("=" * 100)
display(df_all)


# Manager Changes

In [ ]:
# Group by Contact Name - look at leads with multiple deals
contact_deals = Deals.groupby('Contact Name').agg(
    Deal_count=('Id', 'count'),
    Quality=('Quality', lambda x: x.value_counts().index[0] if x.notna().any() else 'Unknown'),
    Managers=('Deal Owner Name', lambda x: x.nunique()),
    Manager_list=('Deal Owner Name', lambda x: list(x.unique())),
    Stage=('Stage', lambda x: list(x.unique())),
    Paid_count=('Stage', lambda x: (x == 'Payment Done').sum())
).reset_index()

# Sort by number of deals
contact_deals = contact_deals.sort_values('Deal_count', ascending=False)

print(f"Contacts with more than 1 deal: {(contact_deals['Deal_count'] > 1).sum()}")
print()
print("Top-20 contacts by number of deals:")
print(contact_deals.head(20))

In [ ]:
# Contacts where managers changed AND quality changed
contact_quality_change = Deals.groupby('Contact Name').agg(
    Deal_count=('Id', 'count'),
    Unique_qualities=('Quality', 'nunique'),
    All_qualities=('Quality', lambda x: list(x.dropna().unique())),
    Managers=('Deal Owner Name', 'nunique'),
    Manager_list=('Deal Owner Name', lambda x: list(x.unique())),
    Paid_count=('Stage', lambda x: (x == 'Payment Done').sum())
).reset_index()

# Filter - only where quality changed
mask_quality_change = contact_quality_change['Unique_qualities'] > 1
print(f"Contacts with a quality change: {mask_quality_change.sum()}")
print()
print(contact_quality_change[mask_quality_change].sort_values('Deal_count', ascending=False).head(20))

In [ ]:
# Detailed view of leads with 2+ deals and a quality change
# Look at the timeline

contacts_3plus = contact_quality_change[
    (contact_quality_change['Deal_count'] >= 2) &
    (contact_quality_change['Unique_qualities'] > 1)
]['Contact Name'].tolist()

print(f"Contacts with 2+ deals and a quality change: {len(contacts_3plus)}")
print()

# Look in detail at each contact
for contact in contacts_3plus[:10]:  # first 10 as an example
    print(f"{'='*60}")
    print(f"Contact: {contact}")
    detail = Deals[Deals['Contact Name'] == contact].sort_values('Created Time')
    print(detail[['Created Time', 'Stage', 'Quality', 'Deal Owner Name', 'Lost Reason']].to_string())
    print()

In [ ]:
# Count contacts where a manager change led to a payment
converted_after_change = contact_quality_change[
    (contact_quality_change['Managers'] > 1) &
    (contact_quality_change['Paid_count'] > 0)
]
print(f"Contacts where manager change → payment: {len(converted_after_change)}")
print(f"Of all contacts with a manager change: {(contact_quality_change['Managers'] > 1).sum()}")
print(f"Manager-change conversion: {len(converted_after_change)/(contact_quality_change['Managers'] > 1).sum()*100:.1f}%")

Findings on the manager-change practice: 1029 contacts went through a manager change. Of these, 86 ultimately paid (8.4% conversion). A manager change yields 8.4% conversion vs. 4.44% for the base average — twice as high! At first glance, the practice works. But there's an important caveat — this means the manager change itself isn't necessarily the cause of the conversion, but rather the higher number of touchpoints with the customer. Perhaps these leads were simply contacted multiple times → more touchpoints → higher conversion chance. These may also be inherently more engaged leads to begin with (since they don't refuse right away). In other words, managers may keep following up with clients after the first consultation, answering additional questions that come up, or suggesting, say, how to get a study grant.
A more experienced manager may have simply reached them at the right moment. Test mechanics: split such leads into two groups — control (standard process) and test (automatic follow-up at set intervals).

In [ ]:
# ── Table 1: Conversion by number of managers ──────────
manager_conv = Deals.groupby('Contact Name').agg(
    Managers=('Deal Owner Name', 'nunique'),
    Deals=('Id', 'count'),
    Paid_count=('Stage', lambda x: (x == 'Payment Done').sum())
).reset_index()

manager_conv['Converted'] = (manager_conv['Paid_count'] > 0).astype(int)

# Group by number of managers
conv_by_managers = manager_conv.groupby('Managers').agg(
    Contacts=('Contact Name', 'count'),
    Paid=('Converted', 'sum')
).reset_index()

conv_by_managers['Conversion_%'] = (
    conv_by_managers['Paid'] / conv_by_managers['Contacts'] * 100
).round(2)

print("Conversion by number of managers:")
print(conv_by_managers)


conv_by_managers.to_parquet(PATH + 'conv_by_managers.parquet', index=False)

# ── Table 2: Conversion by number of touchpoints ──────────────
touches_conv = Deals.groupby('Contact Name').agg(
    Touches=('Id', 'count'),
    Paid_count=('Stage', lambda x: (x == 'Payment Done').sum())
).reset_index()

touches_conv['Converted'] = (touches_conv['Paid_count'] > 0).astype(int)

# Group touches into ranges
def touch_group(n):
    if n == 1:   return '1 touch'
    if n == 2:   return '2 touches'
    if n <= 5:   return '3-5 touches'
    if n <= 10:  return '6-10 touches'
    return '10+ touches'

touches_conv['Touch_group'] = touches_conv['Touches'].apply(touch_group)

conv_by_touches = touches_conv.groupby('Touch_group').agg(
    Contacts=('Contact Name', 'count'),
    Paid=('Converted', 'sum')
).reset_index()

conv_by_touches['Conversion_%'] = (
    conv_by_touches['Paid'] / conv_by_touches['Contacts'] * 100
).round(2)

# Correct order
order = ['1 touch','2 touches','3-5 touches','6-10 touches','10+ touches']
conv_by_touches['Order'] = conv_by_touches['Touch_group'].map(
    {v: i for i, v in enumerate(order)}
)
conv_by_touches = conv_by_touches.sort_values('Order').drop('Order', axis=1)

print("\nConversion by number of touchpoints:")
print(conv_by_touches)


conv_by_touches.to_parquet(PATH + 'conv_by_touches.parquet', index=False)

# ── Table 3: Manager change → payment ─────────────────────
contact_agg = Deals.groupby('Contact Name').agg(
    Managers=('Deal Owner Name', 'nunique'),
    Deals=('Id', 'count'),
    Paid_count=('Stage', lambda x: (x == 'Payment Done').sum()),
    Qualities=('Quality', 'nunique')
).reset_index()

contact_agg = contact_agg[contact_agg['Managers'] > 0]

contact_agg['Manager_change'] = contact_agg['Managers'].apply(
    lambda x: 'Manager changed' if x > 1 else 'Single manager'
)
contact_agg['Converted'] = (contact_agg['Paid_count'] > 0).astype(int)

summary = contact_agg.groupby('Manager_change').agg(
    Contacts=('Contact Name', 'count'),
    Paid=('Converted', 'sum')
).reset_index()

summary['Conversion_%'] = (
    summary['Paid'] / summary['Contacts'] * 100
).round(2)

print("\nComparison: manager changed vs single manager:")
print(summary)


summary.to_parquet(PATH + 'manager_change_conv.parquet', index=False)

print("\nAll files saved!")

# Sample Size

In [ ]:
# full sample size calculation - approach 1
# ============================================================
# Simplified sample size formula (Evan Miller)
# n = 16 * p * (1-p) / Δ²
# where Δ = absolute difference (MDE)
# ============================================================

def sample_size_evan_miller(p, delta):
    """
    p     — baseline conversion (current metric)
    delta — absolute MDE (minimum detectable effect)
    """
    n = 16 * p * (1 - p) / (delta ** 2)
    return int(np.ceil(n))

# ============================================================
# Baseline metrics
# ============================================================
UA           = len(Contacts)
buyers       = paid.drop_duplicates(subset=['Id'])
B            = len(buyers)
C1           = B / UA
AC           = 149523
CAC          = AC / B
leads_month  = UA / 12
buyers_month = B / 12

# ============================================================
# Hypothesis 1 — Increase in C1 conversion (+10% MDE)
# ============================================================
current_c1 = C1
delta_c1   = current_c1 * 0.10   # absolute MDE = 10% of current
new_c1     = current_c1 + delta_c1

n_c1 = sample_size_evan_miller(current_c1, delta_c1)

print("=" * 65)
print("HYPOTHESIS 1 — Increase in C1 conversion")
print("=" * 65)
print(f"Formula:                          n = 16×p×(1-p) / Δ²")
print(f"Current conversion p:             {current_c1:.4f} ({current_c1*100:.2f}%)")
print(f"MDE (Δ = 10% of p):               {delta_c1:.4f} ({delta_c1*100:.2f} p.p.)")
print(f"Expected conversion:              {new_c1:.4f} ({new_c1*100:.2f}%)")
print(f"Minimum sample per group:         {n_c1:,}")
print(f"Total participants (A+B):         {n_c1*2:,}")
print(f"Average lead flow/month:          {leads_month:.0f}")
print(f"Test duration:                    {(n_c1*2)/leads_month:.1f} months")
print()

# ============================================================
# Hypothesis 2 — CAC reduction (-10%)
# A 10% CAC reduction = ~11.1% conversion increase
# (same spend / more buyers)
# ============================================================
current_c1_cac  = C1
new_c1_for_cac  = current_c1_cac / 0.90   # need more buyers
delta_cac       = new_c1_for_cac - current_c1_cac

n_cac = sample_size_evan_miller(current_c1_cac, delta_cac)

print("=" * 65)
print("HYPOTHESIS 2 — CAC reduction by 10%")
print("=" * 65)
print(f"Formula:                          n = 16×p×(1-p) / Δ²")
print(f"Current CAC:                      €{CAC:.0f}")
print(f"Target CAC (-10%):                €{CAC*0.9:.0f}")
print(f"Current conversion p:             {current_c1_cac:.4f} ({current_c1_cac*100:.2f}%)")
print(f"Required conversion:              {new_c1_for_cac:.4f} ({new_c1_for_cac*100:.2f}%)")
print(f"MDE (Δ):                          {delta_cac:.4f} ({delta_cac*100:.2f} p.p.)")
print(f"Minimum sample per group:         {n_cac:,}")
print(f"Total participants (A+B):         {n_cac*2:,}")
print(f"Average lead flow/month:          {leads_month:.0f}")
print(f"Test duration:                    {(n_cac*2)/leads_month:.1f} months")
print()

# ============================================================
# Summary table
# ============================================================
summary = pd.DataFrame({
    'Hypothesis':  ['H1: C1 increase', 'H2: CAC reduction'],
    'Metric':      [f'C1: {current_c1*100:.2f}% → {new_c1*100:.2f}%',
                   f'CAC: €{CAC:.0f} → €{CAC*0.9:.0f}'],
    'MDE (Δ)':     [f'{delta_c1*100:.2f} p.p.',
                   f'{delta_cac*100:.2f} p.p.'],
    'Group A':     [n_c1, n_cac],
    'Group B':     [n_c1, n_cac],
    'Total':       [n_c1*2, n_cac*2],
    'Duration (mo)': [
        round((n_c1*2) / leads_month, 1),
        round((n_cac*2) / leads_month, 1)
    ]
})

print("=" * 80)
print("SUMMARY TABLE")
print("=" * 80)
print(summary.to_string(index=False))

Two approaches were used to assess the feasibility of running an A/B test.

The first approach is based on the classic minimum sample size formula for a pre-defined minimum detectable effect (MDE = 10%). This calculation shows how much data is needed to statistically reliably detect changes at a given significance level and test power.

The second approach is based on the company's actually available data volume. Here, the minimum detectable effect (MDE) was calculated for the real number of leads that can be collected in two weeks. This approach helps assess what changes in metrics the company would actually be able to detect under real experiment conditions.

In [ ]:
# ============================================================
# Actually available sample size
# ============================================================

test_weeks = 2

available_leads = round(leads_month * test_weeks / 4)
available_buyers = round(buyers_month * test_weeks / 4)

print("="*70)
print("ACTUALLY AVAILABLE SAMPLE SIZE FOR THE A/B TEST")
print("="*70)

print(f"Test duration:                  {test_weeks} weeks")
print(f"Leads over the period:          {available_leads*2}")
print(f"Leads per group:                {available_leads}")
print(f"Buyers over the period:         {available_buyers*2}")
print(f"Buyers per group:               {available_buyers}")

# ============================================================
# Minimum Detectable Effect (MDE)
# ============================================================

def min_detectable_effect(n, p,
                          alpha=0.05,
                          power=0.80):

    z_alpha = norm.ppf(1-alpha/2)
    z_beta  = norm.ppf(power)

    return (
        (z_alpha + z_beta)
        * np.sqrt(2*p*(1-p)/n)
    )

mde_c1 = min_detectable_effect(
    available_leads,
    current_c1
)

print()
print("="*70)
print("MINIMUM DETECTABLE EFFECT (MDE)")
print("="*70)

print(f"H1 — Conversion")
print(f"Current C1:                    {current_c1*100:.2f}%")
print(f"Minimum detectable increase:   +{mde_c1*100:.2f} p.p.")
print(f"I.e. the test can detect a change up to about {(current_c1+mde_c1)*100:.2f}%")

print()

print(f"H2 — CAC")
print("Since CAC depends on conversion,")
print(f"the minimum detectable conversion increase will also be +{mde_c1*100:.2f} p.p.")

Thus, the first calculation shows how many participants are needed to detect a pre-defined effect, while the second shows what effect can actually be detected given the available sample size. Using both approaches makes it possible to compare the theoretical requirements of statistics with the company's practical capabilities.